# Example 11.3 / 3: Trace What the Solver Receives

Companion notebook for **11.3 Nonlinear Expressions and Functions** in the AMPL lab Documentation, section "Trace What the Solver Receives". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Trace What the Solver Receives

For MP piecewise-linear approximation, `cvt:plapprox:reltol` controls relative approximation
accuracy and `cvt:plapprox:domain` controls the fallback argument domain. Tight, valid bounds can
reduce approximation size and improve numerical behavior. An approximation is not an exact
reformulation, so validate the returned point in the original nonlinear expressions and state the
approximation policy when reporting results. Automatic Piecewise-Linear Approximation (AMPL lab documentation, section 10.4.3)
develops that workflow, and How MP Solvers Reformulate Your Model (AMPL lab documentation, section 8.6) explains the broader
transformation boundary.

Both options on $\min\ e^x-2x$, whose exact optimum is $x=\log2\approx0.693147$ with value
$2-2\log2\approx0.613706$. Tightening `cvt:plapprox:reltol` from its default `0.01` to `0.0001`
moves the HiGHS point from about 0.6087 to 0.7054 and the objective, evaluated by AMPL in the
original function, from 0.620641 to 0.613857:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= -2, <= 3;  # finite bounds limit the approximation domain

minimize Obj: exp(x) - 2 * x;  # exact optimum at x = log 2

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for opts in ("", "cvt:plapprox:reltol=0.0001"):
    ampl.option["highs_options"] = opts
    ampl.solve(solver="highs", verbose=False)
    print(repr(opts), ampl.solve_result, round(ampl.var["x"].value(), 4),
          round(ampl.obj["Obj"].value(), 6))

'' solved 0.6087 0.620641
'cvt:plapprox:reltol=0.0001' solved 0.7054 0.613857
